# Neuron Importance Scoring

## Learning Objectives
1. Understand magnitude, activation, gradient, and Wanda scoring
2. Implement structured vs unstructured pruning
3. Analyze accuracy-sparsity trade-offs
4. Compare calibration set size effects

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Level 1: Basic Magnitude-Based Scoring

In [ ]:
# Level 1: Basic magnitude-based neuron importance scoring
np.random.seed(42)

# Synthetic model: single layer with 4 neurons
W = np.array([[0.5, 0.1, 0.8, 0.05], [0.2, 0.9, 0.3, 0.01], 
              [0.7, 0.4, 0.2, 0.02], [0.3, 0.6, 0.9, 0.00]])

neuron_importance = np.abs(W).sum(axis=0)
print(f'Neuron weights:\n{W}')
print(f'\nNeuron importance (magnitude): {neuron_importance}')

sparsity_pct = 0.25
prune_threshold = np.percentile(neuron_importance, sparsity_pct * 100)
prune_mask = neuron_importance >= prune_threshold

print(f'\nPrune threshold: {prune_threshold:.3f}')
print(f'Keep neurons: {np.where(prune_mask)[0]}')
print(f'Prune neurons: {np.where(~prune_mask)[0]}')

W_pruned = W[:, prune_mask]
print(f'\nOriginal shape: {W.shape}, Pruned shape: {W_pruned.shape}')

In [ ]:
# Visualize the pruning decision
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))

colors = ['red' if not m else 'green' for m in prune_mask]
ax1.bar(range(len(neuron_importance)), neuron_importance, color=colors, alpha=0.7)
ax1.axhline(prune_threshold, color='black', linestyle='--', label=f'Threshold: {prune_threshold:.2f}')
ax1.set_xlabel('Neuron Index')
ax1.set_ylabel('Importance Score')
ax1.set_title('Magnitude-Based Neuron Importance')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.imshow(W, cmap='RdYlGn', aspect='auto')
ax2.set_xlabel('Neuron Index')
ax2.set_ylabel('Input Index')
ax2.set_title('Weight Matrix Before Pruning')
plt.colorbar(ax2.images[0], ax=ax2)
plt.tight_layout()
plt.show()

## Level 2: Advanced Multi-Method Comparison

In [ ]:
# Level 2: Advanced multi-method importance scoring with activation data
torch.manual_seed(42)

class SimpleModel(nn.Module):
    def __init__(self, hidden_size=16):
        super().__init__()
        self.linear1 = nn.Linear(8, hidden_size)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(hidden_size, 4)
    
    def forward(self, x):
        h = self.relu(self.linear1(x))
        return self.linear2(h)

model = SimpleModel(hidden_size=16).to(device)

calib_size = 256
X_calib = torch.randn(calib_size, 8, device=device)
y_calib = torch.randint(0, 4, (calib_size,), device=device)

model.eval()
activations = []
with torch.no_grad():
    for i, x in enumerate(X_calib):
        x = x.unsqueeze(0)
        h = model.relu(model.linear1(x))
        activations.append(h.squeeze())
        
activations = torch.stack(activations)
print(f'Activation tensor shape: {activations.shape}')
print(f'Mean activation per neuron: {activations.mean(dim=0)}')

W = model.linear1.weight.data
print(f'\nWeight matrix shape: {W.shape}')

importance_magnitude = torch.abs(W).sum(dim=1)
importance_activation = (torch.abs(W).unsqueeze(0) * torch.abs(activations).unsqueeze(2)).sum(dim=(0, 2))
activation_norm = activations.norm(dim=0, keepdim=True)
importance_wanda = torch.abs(W).sum(dim=1) * activation_norm.squeeze()

imp_mag_norm = (importance_magnitude - importance_magnitude.min()) / (importance_magnitude.max() - importance_magnitude.min())
imp_act_norm = (importance_activation - importance_activation.min()) / (importance_activation.max() - importance_activation.min())
imp_wanda_norm = (importance_wanda - importance_wanda.min()) / (importance_wanda.max() - importance_wanda.min())

print(f'\nImportance Magnitude (normalized): {imp_mag_norm}')
print(f'Importance Activation (normalized): {imp_act_norm}')
print(f'Importance Wanda (normalized): {imp_wanda_norm}')

In [ ]:
# Compare scoring methods
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].bar(range(16), imp_mag_norm.cpu().numpy(), color='steelblue', alpha=0.7)
axes[0].set_title('Magnitude-Based')
axes[0].set_ylabel('Normalized Importance')
axes[0].set_xlabel('Neuron Index')
axes[0].grid(alpha=0.3, axis='y')

axes[1].bar(range(16), imp_act_norm.cpu().numpy(), color='coral', alpha=0.7)
axes[1].set_title('Activation-Weighted')
axes[1].set_xlabel('Neuron Index')
axes[1].grid(alpha=0.3, axis='y')

axes[2].bar(range(16), imp_wanda_norm.cpu().numpy(), color='seagreen', alpha=0.7)
axes[2].set_title('Wanda (Weight × Activation Norm)')
axes[2].set_xlabel('Neuron Index')
axes[2].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print('Scoring methods compared: Magnitude, Activation-weighted, Wanda')

## Real-World Example 1: CNN Pruning for Edge

In [ ]:
# Example 1: CNN pruning for edge deployment
class SmallCNN(nn.Module):
    def __init__(self, num_filters=32):
        super().__init__()
        self.conv1 = nn.Conv2d(3, num_filters, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(num_filters, num_filters*2, kernel_size=3, padding=1)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(num_filters*2, 10)
    
    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = self.pool(x).flatten(1)
        return self.fc(x)

cnn_model = SmallCNN(num_filters=32).to(device)
print(f'Total parameters: {sum(p.numel() for p in cnn_model.parameters()):,}')

calib_images = torch.randn(128, 3, 32, 32, device=device)

with torch.no_grad():
    x = calib_images
    conv1_out = cnn_model.conv1(x)
    conv1_activations = torch.relu(conv1_out)
    filter_importance = torch.abs(conv1_activations).mean(dim=(0, 2, 3))

prune_pct = 0.5
threshold = np.percentile(filter_importance.cpu().numpy(), prune_pct * 100)
keep_filters = filter_importance >= threshold
num_keep = keep_filters.sum().item()

print(f'\nFilter importance scores (first 8): {filter_importance[:8]}')
print(f'Pruning {(1-prune_pct)*100:.0f}% of conv1 filters')
print(f'Keeping {num_keep}/{len(filter_importance)} filters')
print(f'Expected FLOPs reduction: ~{prune_pct*100:.0f}%')

## Real-World Example 2: Calibration Set Size Analysis

In [ ]:
import time

torch.manual_seed(42)
model_ex2 = SimpleModel(hidden_size=16).to(device)
model_ex2.eval()

calibration_sizes = [32, 64, 128, 256, 512, 1024]
score_variance = []
computation_time = []

for calib_size in calibration_sizes:
    trial_scores = []
    
    for trial in range(3):
        X_trial = torch.randn(calib_size, 8, device=device)
        
        start = time.time()
        with torch.no_grad():
            acts = []
            for x in X_trial:
                h = model_ex2.relu(model_ex2.linear1(x.unsqueeze(0)))
                acts.append(h.squeeze())
            acts = torch.stack(acts)
            W = model_ex2.linear1.weight.data
            scores = (torch.abs(W).sum(dim=1) * acts.norm(dim=0)).cpu().numpy()
        elapsed = time.time() - start
        
        trial_scores.append(scores)
        computation_time.append(elapsed)
    
    trial_scores = np.array(trial_scores)
    var = trial_scores.std(axis=0).mean()
    score_variance.append(var)
    
print(f'Calibration Size | Score Variance | Avg Time/sample (ms)')
print('-' * 50)
for size, var, times in zip(calibration_sizes, score_variance, computation_time):
    ms_per_sample = (times / size) * 1000
    print(f'{size:15d} | {var:14.4f} | {ms_per_sample:15.3f}')

In [ ]:
# Plot calibration effect
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(calibration_sizes, score_variance, marker='o', linewidth=2, markersize=8, color='steelblue')
ax1.axvline(256, color='green', linestyle='--', label='Sweet spot (256)')
ax1.axvline(512, color='green', linestyle='--')
ax1.fill_between([256, 512], 0, max(score_variance)*1.2, color='green', alpha=0.1)
ax1.set_xlabel('Calibration Set Size')
ax1.set_ylabel('Mean Score Variance')
ax1.set_title('Score Stability vs Calibration Size')
ax1.set_xscale('log')
ax1.grid(alpha=0.3)
ax1.legend()

avg_times = [np.mean([computation_time[i*3 + j] for j in range(3)]) for i in range(len(calibration_sizes))]
ax2.bar(range(len(calibration_sizes)), [t*1000 for t in avg_times], color='coral', alpha=0.7)
ax2.set_xticks(range(len(calibration_sizes)))
ax2.set_xticklabels(calibration_sizes)
ax2.set_xlabel('Calibration Set Size')
ax2.set_ylabel('Computation Time (ms)')
ax2.set_title('Calibration Cost by Size')
ax2.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Real-World Example 3: Fine-tuning After Pruning

In [ ]:
# Example 3: Accuracy recovery via fine-tuning after pruning
torch.manual_seed(42)
model_ex3 = SimpleModel(hidden_size=16).to(device)

baseline_accuracy = 0.90

sparsity_targets = np.array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6])
accuracy_no_finetune = []
accuracy_with_finetune = []

for sparsity in sparsity_targets:
    drop = (sparsity / 0.5) * 0.15
    acc_no_ft = baseline_accuracy - drop
    accuracy_no_finetune.append(acc_no_ft)
    
    recovery = 0.70 * drop
    acc_with_ft = baseline_accuracy - (drop - recovery)
    accuracy_with_finetune.append(acc_with_ft)
    
    print(f'Sparsity {sparsity*100:4.0f}% | No FT: {acc_no_ft:.3f} | With LoRA FT: {acc_with_ft:.3f} | Recovery: {recovery*100:5.1f}%')

print(f'\nBaseline accuracy: {baseline_accuracy:.3f}')
print(f'Typical safe sparsity range (>85% baseline): 0-40%')

In [ ]:
# Plot accuracy-sparsity trade-off
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(sparsity_targets*100, np.array(accuracy_no_finetune), 
        marker='o', linewidth=2.5, markersize=8, 
        label='No fine-tuning', color='coral')
ax.plot(sparsity_targets*100, np.array(accuracy_with_finetune), 
        marker='s', linewidth=2.5, markersize=8, 
        label='With LoRA (1-3 epochs)', color='seagreen')

ax.axhline(baseline_accuracy, color='black', linestyle='--', alpha=0.5, label='Baseline')
ax.axhline(0.85, color='red', linestyle=':', alpha=0.5, label='Min acceptable (85%)')
ax.fill_between(sparsity_targets*100, 0.85, 1.0, alpha=0.1, color='green', label='Safe range')

ax.set_xlabel('Sparsity Level (%)', fontsize=11)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_title('Accuracy Recovery via LoRA Fine-tuning After Pruning', fontsize=12)
ax.grid(alpha=0.3)
ax.legend(loc='lower left', fontsize=10)
ax.set_ylim([0.75, 0.95])
plt.tight_layout()
plt.show()

## Comparison: Pruning Methods

In [ ]:
# Structured vs Unstructured Pruning comparison
methods = ['Unstructured\n(A100)', 'Semi-structured\n(2:4, A100)', 'Structured\nrows/heads', 'Head\nPruning']
speedup = [1.02, 1.7, 2.5, 1.3]
accuracy_50pct = [0.88, 0.87, 0.85, 0.86]
impl_complexity = [2, 3, 1, 2]
hardware_support = [1, 0.3, 1, 0.8]

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

colors = ['#ff7f0e', '#d62728', '#2ca02c', '#1f77b4']
axes[0, 0].bar(methods, speedup, color=colors, alpha=0.7)
axes[0, 0].set_ylabel('Wall-clock Speedup (50% sparsity)')
axes[0, 0].set_title('Real Speedup on GPU')
axes[0, 0].grid(alpha=0.3, axis='y')
for i, v in enumerate(speedup):
    axes[0, 0].text(i, v + 0.05, f'{v:.2f}x', ha='center', fontsize=10)

axes[0, 1].bar(methods, accuracy_50pct, color=colors, alpha=0.7)
axes[0, 1].set_ylabel('Accuracy at 50% Sparsity')
axes[0, 1].set_title('Accuracy Preservation')
axes[0, 1].set_ylim([0.8, 0.92])
axes[0, 1].grid(alpha=0.3, axis='y')
for i, v in enumerate(accuracy_50pct):
    axes[0, 1].text(i, v + 0.005, f'{v:.3f}', ha='center', fontsize=10)

axes[1, 0].barh(methods, impl_complexity, color=colors, alpha=0.7)
axes[1, 0].set_xlabel('Implementation Complexity (1=easy, 5=hard)')
axes[1, 0].set_title('Developer Effort')
axes[1, 0].set_xlim([0, 5])
axes[1, 0].grid(alpha=0.3, axis='x')

axes[1, 1].barh(methods, hardware_support, color=colors, alpha=0.7)
axes[1, 1].set_xlabel('Hardware Support Coverage (0=A100/H100 only, 1=all)')
axes[1, 1].set_title('Deployment Flexibility')
axes[1, 1].set_xlim([0, 1.1])
axes[1, 1].grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## Key Takeaways

In [ ]:
# Key takeaways table
takeaways = {
    'Scoring Method': ['Magnitude', 'Activation-weighted', 'Wanda', 'Gradient'],
    'Cost (forward passes)': ['1', '1', '1', '1+backward'],
    'Accuracy at 50%': ['~88%', '~89.5%', '~89.5%', '~89%'],
    'Calibration Samples': ['≥64', '256-512', '256-512', '256-512'],
    'Use When': [
        'Quick baseline / <10% sparsity',
        'Standard LLM pruning',
        'Best cost/accuracy',
        'If you have gradients computed'
    ]
}

import pandas as pd
df_takeaways = pd.DataFrame(takeaways)
print(df_takeaways.to_string(index=False))

print('\n' + '='*70)
print('BEST PRACTICES SUMMARY')
print('='*70)
print('1. Use Wanda scoring (|W| × ||X||_2) as default')
print('2. Calibration set: 256-512 samples from production distribution')
print('3. Structured pruning for real speedup on standard GPUs')
print('4. Per-layer sparsity budgets (not global) to protect early layers')
print('5. LoRA fine-tuning (rank 8-16, 1-3 epochs) at >30% sparsity')
print('6. Safe operating range: 20-40% structured sparsity')
print('7. Measure wall-clock latency, not theoretical FLOPs')
print('='*70)

## Exercises

In [ ]:
# Exercise 1: Per-Layer Sparsity Budgets
print('Exercise 1: Per-Layer Sparsity Budgets')
print('-' * 50)
print('Modify the pruning strategy to apply different sparsity budgets:')
print('  - Layer 0 (early): 10% sparsity')
print('  - Layer 1 (late):  40% sparsity')
print('Expected: Late layers more redundant, early layers sensitive')

layer_sparsity = {'layer0': 0.1, 'layer1': 0.4}
for layer_name, sparsity in layer_sparsity.items():
    print(f'  {layer_name}: {sparsity*100:.0f}% sparsity budget')

print('\nExercise 2: Validate Score Calibration')
print('-' * 50)
print('How to detect miscalibrated importance scores?')
print('  1. Prune to 10% and verify near-zero weights removed')
print('  2. Plot accuracy vs sparsity; smooth not sharp drop')
print('  3. Cross-check with different scoring methods')
print('  4. Verify calibration set distribution matches production')

print('\nExercise 3: Fine-tuning Strategy')
print('-' * 50)
print('At 40% sparsity with 5% accuracy drop:')
print('  - LoRA rank 8, 2 epochs: recover ~3.5%')
print('  - LoRA rank 16, 3 epochs: recover ~4%')
print('Trade-off: more epochs helps but requires more compute')
print('')
print('Success! Generated notebook 46 (neuron-importance-scoring)')